# H3 Compression (text reports only, for now)

Frozen before the run.

**Claim.** With a compressed payload, split questions fail because a worker can't tell which of its facts another worker's hop needs.

**Arms** (thinking off; 300 per cell, seed 0):
- K0 (uncompressed reference).
- B1.
- T1 with each report capped at 256 or 1024 tokens.

**Prediction.**
- At 256 tokens, K0 − T1 is larger under split placements than under AA/AAA.
- At 1024 the reports are close to lossless on the toy, so T1 ≈ B1 there.
  - Text gets re-read jointly by the orchestrator, so T1 can beat K0. It did in the pilot.
- On MuSiQue, T1's gap to B1 grows with hops.

**Decision rule.** The compression claim needs (K0 − T1-256 | split) − (K0 − T1-256 | colo) > 0, with the CI excluding 0. If T1 beats K0 everywhere, compression isn't the dominant loss at these budgets, and the separate-vs-joint encoding axis (H1) is.

**Pilot (Qwen2.5-7B, forced reasoning, uncapped reports):**

| Setting | Single | K0 | T1 |
|---|---|---|---|
| toy, 1 crossing | 99 | 85 | 100 (598 tok/q) |
| toy, 2 crossings | 97 | 32 | 49 (722 tok/q) |
| MuSiQue 2-step, F1 | .414 | .294 | .387 |
| MuSiQue 3-step, F1 | .415 | .313 | .314 |

**Not built yet:**
- T2 (copied sentences), K3 and K4 (as KV and re-prefilled).
- The hop-1 vs hop-2 retention measure.
- Budgets of 16 and 64.

In [ ]:
# Settings. Headless full run: see README "Running".
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SMOKE = True        # True: 2 questions per cell, to read raw outputs first. False: the full design.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from latent_dispatch import analysis, arms, data
from latent_dispatch.model import Model
from latent_dispatch.results import load_results, results_path, run
from latent_dispatch.score import contains_answer, normalize

## Questions and arms

Reports are ~600 generated tokens per worker, so 300 questions per (dataset, qtype, placement) cell, seed 0.

In [ ]:
qs = data.load("toy", seed=0) + data.load("toy_d0", seed=0) + data.load("musique", seed=0)
qs = data.per_cell(qs, 2 if SMOKE else 300)
print(len(qs), "questions")

ARMS = {
    "k0": (arms.k0, {}),
    "b1": (arms.b1, {}),
    "t1_256": (arms.t1, {"report_tokens": 256}),
    "t1_1024": (arms.t1, {"report_tokens": 1024}),
}

In [ ]:
model = Model.load(MODEL, dtype=DTYPE)
for name, (arm, options) in ARMS.items():
    run(model, qs, name, arm, **options)

In [ ]:
df = load_results(qs, ARMS, MODEL, DTYPE)
if SMOKE:
    analysis.peek(df)

## Accuracy

In [ ]:
analysis.accuracy(df)

## K0 − T1 and B1 − T1, and whether K0 − T1 grows with crossings

In [ ]:
for t in ("t1_256", "t1_1024"):
    for ref in ("k0", "b1"):
        print(f"{ref} − {t}")
        display(analysis.paired(df, ref, t))
    for ds in ("toy_d2", "musique"):
        print(f"{ds}: (k0 − {t}) at c crossings minus at 0")
        display(analysis.crossing_trend(df, "k0", t, ds))

## Report length and truncation

In [ ]:
for t in ("t1_256", "t1_1024"):
    rows = df[df.arm == t]
    if len(rows):
        tokens = [n for ns in rows.worker_tokens for n in ns]
        cut = [x for xs in rows.truncated for x in xs]
        print(f"{t}: mean report tokens {sum(tokens) / len(tokens):.0f}, truncated {100 * sum(cut) / len(cut):.1f}%")

## Findings

_(fill in after the full run)_